# KAT-VAD v5 — **Stage 2**: train R3 / R5 / R5Δ × 3 seeds, read S2-7 … S2-10

Rules: `core/docs/v5/PREREG_V5.md` **§2** (signed 2026-10-27) + **§2.4** (B1–B10). Stage 1 must have passed every
gate (`Thesis-V5/outputs/v5/v5_stage1/REPORTS/stage1_summary.md`); step 1 refuses to run otherwise. R1 is Stage 1's
(three seeds, v5 path, B9); R0 = v2 Nexar A0 is printed only. Nothing here changes a rule after a number.

| step | what | gate / read | GPU? |
|---|---|---|---|
| 1 | setup + Stage 1 gates re-read | all PASS | — |
| 2 | **train** R3, R5, R5Δ (`R5d`) × s2024–2026 (A2's recipe + `v5.*`, 1,749 steps each) | **S1-G4** J10 + config | yes |
| 3 | **S1-G6** no collapse (read before S2-7) | finite losses; entropy printed | no |
| 4 | `nexar_val` protocol N-B, R1 / R3 / R5 / R5Δ (S2-9 guardrail) | printed | yes |
| 5 | **DoTA-CAP-dev** protocol B, R1 / R3 / R5 / R5Δ (S2-7) | — | yes |
| 6 | maps: dump R3 / R5 / R5Δ × 3 seeds on CAP-dev, then **S2-8** read | — | yes / no |
| 7 | **read-out** S2-7 / S2-9 / S2-10 → **paste `REPORTS/stage2_readout.md` + `maps_readout.md` to Claude** | decision | no |
| 8 | **S2-11** `nexar_test` once, R1 + the adopted row (only after step 7; off by default) | printed | yes |

**S2-12 cost (estimate):** 9 training runs × 1,749 steps (input rows ≈ 50× v2's: every row carries 49 × 384 grid
values, R5Δ twice that); `nexar_val` grids ≈ 10 GB staged to the VM once; 12 `nexar_eval` + 4 `protocol_b_eval` +
9 map dumps. Every step skips work already verified on Drive, so the notebook resumes after a disconnect.

**Drive:** code `Thesis-V5/kat-vad` (re-upload the v5 branch first); writes only to
`Thesis-V5/outputs/v5/v5_stage2/`; reads Stage 1 (`v5_stage1/`), the v5 grids, and `Thesis/` + `Thesis-V2/`
read-only.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (DoTA), cache/clip, ckpts
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: v2 caches / runs / reads, Stage 0 grids
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # code + Stage 1 + everything Stage 2 writes
os.environ['PROJECT_ROOT']       = DRIVE_V5
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V5}/outputs'

os.environ['REPO'] = f'{DRIVE_V5}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v5 branch there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V5_STAGE2_DIRNAME'), 'this upload predates v5 Stage 2 -- re-upload the code'
assert (REPO / 'core/tools/v5_stage2.py').is_file(), 'no core/tools/v5_stage2.py in this upload'
prereg = (REPO / 'core/docs/v5/PREREG_V5.md').read_text(encoding='utf-8')
assert '**Status: SIGNED' in prereg.split('## §2 Stages 1–2', 1)[1], 'PREREG_V5 §2 is not signed in this upload'
assert '**B10' in prereg, 'upload has no §2.4 B10 (Stage 2 read-out notes)'

ENC      = constants.V5_ENCODER                                   # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW                    # 'window'
DATASET  = constants.NEXAR_DATASET
CONTROL  = constants.V5_CONTROL_ROW                               # R1 (Stage 1)
NEW_ROWS = list(constants.V5_STAGE2_ROWS)                         # R3, R5, R5d
ALL_ROWS = [CONTROL, *NEW_ROWS]
MAP_ROWS = NEW_ROWS                                               # R1's map is uniform (S2-8)
SEEDS    = list(constants.V5_RUN_SEEDS)                           # 2024-2026
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV

V2_CACHE = Path(DRIVE_V2) / 'cache'
V2_OUT   = Path(DRIVE_V2) / 'outputs'
V5_CACHE = Path(DRIVE_V5) / 'cache'
RAW      = constants.DATA_ROOT / DATASET / 'raw'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
VIDEO_S1 = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
A2_BAKE  = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                  # v2 Nexar A2's input + statistics
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
GRIDS    = V5_CACHE / constants.V5_GRID_CACHE_DIRNAME / ENC
TRAIN_GRID = GRIDS / f'{DATASET}_s{constants.V5_NEXAR_TRAIN_STRIDE}_grid{constants.V5_GRID_COARSE}'
VAL_GRID   = GRIDS / f'{DATASET}_s{constants.V5_NEXAR_VAL_STRIDE}_grid{constants.V5_GRID_COARSE}'
TEST_GRID  = GRIDS / f'{DATASET}_test_s{constants.V5_NEXAR_VAL_STRIDE}_grid{constants.V5_GRID_COARSE}'  # S2-11 only
DOTA_GRID  = V2_CACHE / constants.V5_GRID_CACHE_DIRNAME / ENC / constants.V5_GRID_DATASET_TAG   # Stage 0 (read-only)
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
DOTA_A2  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'                     # A2 rows, Nexar stats
V2_NEXAR = V2_OUT / constants.NEXAR_FULL_RUNS_DIR                                       # v2 Nexar runs + reads
A2_RUNS  = V2_NEXAR / 'A2'
R0_PB    = V2_NEXAR / 'eval' / 'dota' / 'A0' / CAP_DEV / 'protocol_b_readout.json'     # printed (S2-14)
A2_PB    = V2_NEXAR / 'eval' / 'dota' / 'A2' / CAP_DEV / 'protocol_b_readout.json'     # printed (S2-14)
PRIOR    = V2_OUT / 'v2_e3' / 'REPORTS' / 'position_prior' / CAP_DEV / 'position_prior.npz'   # p_T2 (printed)
STAGE0   = V2_OUT / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE0_DIRNAME
STAGE1   = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE1_DIRNAME   # R1 runs + gates
ROOT     = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE2_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/v5s2')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
assert constants.OUTPUT_ROOT == Path(DRIVE_V5) / 'outputs', constants.OUTPUT_ROOT

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']                     # 1,749 = 33 x 53 (v2 §29)
for name, path in (('Nexar_window', DATA), ('census', CENSUS), ('clip s1', CLIP_S1), ('V2-S s1', VIDEO_S1),
                   ('A2 bake', A2_BAKE), ('KNN', KNN), ('train grids', TRAIN_GRID),
                   ('head', TRAIN_GRID / constants.V5_HEAD_FILENAME), ('val grids', VAL_GRID),
                   ('DoTA grids', DOTA_GRID), ('DoTA s1', DOTA_S1), ('DoTA-CAP-dev A2 bake', DOTA_A2),
                   ('v2 A2 runs', A2_RUNS), ('R0 read', R0_PB), ('A2 read', A2_PB), ('p_T2', PRIOR),
                   ('Stage 1', STAGE1)):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}   OK')

# Stage 1 must have passed every gate (PREREG_V5 §2.2: any failure -> nothing in Stage 2 is trained or read)
S1_REPORTS = STAGE1 / 'REPORTS'
for gname in ('g1_extract_s8', 'g1_extract_s1', 'g2_init', 'g5_maps', 'g4_provenance'):
    r = json.loads((S1_REPORTS / f'{gname}.json').read_text())
    assert r['pass'], f'Stage 1 {gname} did not pass -- Stage 2 must not run (§2.2)'
    print(f'  Stage 1 {gname}: PASS')
print(f'plan: {EPOCHS} epochs = {STEPS} steps | writes -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 2, 6, 7 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'epoch' in line and 'batch' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def gate(name):
    '''Read a gate's json from REPORTS and stop the notebook on FAIL (PREREG_V5 §2.2).'''
    r = json.loads((REPORTS / f'{name}.json').read_text())
    print((REPORTS / f'{name}.md').read_text())
    assert r['pass'], f'{name} FAILED -- stop here (PREREG_V5 §2.2); paste {name}.md to Claude'
    return r


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')

## 2. Train R3, R5, R5Δ × 3 seeds, then **S1-G4** (GPU)

Exactly Stage 1's command for R1 (v2 Nexar A2's recipe + `--grid-dir`), with the row's `v5.*` keys from
`constants.V5_ROWS` (R3: 1 query, no sMIL; R5: 8 queries + sMIL λ 0.5; R5d: R5 with change keys). Local training,
atomic publish to Drive, re-verified after a flush (lesson (bi)). A run verified on Drive is never retrained.

In [ ]:
L_DATA = stage(DATA, LOCAL / 'stage' / f'data_{C}')
L_A2   = stage(A2_BAKE, LOCAL / 'stage' / 'A2_s8')
L_GRID = stage(TRAIN_GRID, LOCAL / 'stage' / 'grid_s8')
L_KNN  = stage(KNN, LOCAL / 'stage' / f'knn_{C}.npz')
assert (L_GRID / constants.V5_HEAD_FILENAME).is_file(), 'the staged train grids have no v5_head.npz'


def train_cmd(row, seed, out_dir):
    v5 = [x for k, v in constants.V5_ROWS[row].items() for x in ('--set', f'v5.{k}={v}')]
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={seed}',
            '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
            '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
            '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
            '--set', 'data.is_egocentric=false',
            '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}', *v5,
            '--data-dir', L_DATA, '--clip-dir', L_A2, '--knn-cache', L_KNN, '--grid-dir', L_GRID,
            '--output-dir', out_dir]


def run_dir(row, seed):
    return (STAGE1 if row == CONTROL else ROOT) / row / f's{seed}'


def verified(d):
    last = metrics_last(d / 'metrics.jsonl')
    return bool(last) and ckpt_step(d / 'checkpoint_last.pt') == last['global_step'] == STEPS


def train_row(row, seed):
    drive_dir, local = ROOT / row / f's{seed}', LOCAL / 'runs' / row / f's{seed}'
    if verified(drive_dir):
        print(f'{row} s{seed}: verified on Drive -- not retrained')
        return drive_dir
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run(train_cmd(row, seed, local) + resume, log=LOCAL / f'train_{row}_s{seed}.log', quiet_steps=True)
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row} s{seed}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{row} s{seed}: Drive copy is not step {STEPS}'
    print(f'{row} s{seed}: {(time.time() - t0) / 60:.0f} min, last mil {last.get("mil")}, smil {last.get("smil")}')
    return drive_dir


for seed in SEEDS:                     # seed-major: a disconnect leaves complete seeds, not complete rows
    for row in NEW_ROWS:
        train_row(row, seed)
for seed in SEEDS:
    assert verified(run_dir(CONTROL, seed)), f'Stage 1 R1 s{seed} is not a verified run'
run([sys.executable, '-m', 'core.tools.v5_stage1', 'provenance', '--root', ROOT, '--a2-root', A2_RUNS,
     '--rows', *NEW_ROWS, '--seeds', *SEEDS, '--steps', STEPS, '--out-dir', REPORTS])
gate('g4_provenance')

## 3. **S1-G6** no collapse (read before S2-7)

Last-step `mil` / `smil` finite (a non-finite loss stops the notebook). Query entropy is printed; a run whose every
query stays ≥ 0.999 log 49 is reported as "did not leave the mean" — still read, not a failure.

In [ ]:
run([sys.executable, '-m', 'core.tools.v5_stage1', 'collapse', '--root', ROOT, '--rows', *NEW_ROWS,
     '--seeds', *SEEDS, '--out-dir', REPORTS])
gate('g6_collapse')

## 4. `nexar_val` — protocol N-B per row (S2-9 guardrail, GPU)

`nexar_eval` with the three seeds of a row (seed-averaged per frame), A2's statistics, the `s1` grids (§2.4 B1:
crops start on any native phase). The guardrail reads each run's own crop endpoint (`per_run_crop_endpoint`).

In [ ]:
from core.data.v2_splits import load_split

VAL_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_VAL))
L_VAL_X = stage(CLIP_S1, LOCAL / 'stage' / 'clip_s1_val', VAL_IDS)
L_VAL_U = stage(VIDEO_S1, LOCAL / 'stage' / 'video_s1_val', VAL_IDS)
L_VAL_G = stage(VAL_GRID, LOCAL / 'stage' / 'grid_s1_val', VAL_IDS)    # ~10 GB, once


def run_args(row):
    return [x for s in SEEDS for x in ('--run', f's{s}', run_dir(row, s) / 'checkpoint_last.pt')]


NEXAR_OUT = ROOT / 'eval' / 'nexar_val'
for row in ALL_ROWS:
    out = NEXAR_OUT / row
    if (out / 'nexar_readout.json').exists():
        print(f'nexar_val {row}: already read')
        continue
    local, t0 = LOCAL / 'nexar_val' / row, time.time()
    run([sys.executable, '-m', 'core.tools.nexar_eval', *run_args(row), '--census', CENSUS,
         '--clip-s1-dir', L_VAL_X, '--video-s1-dir', L_VAL_U, '--stats-dir', L_A2, '--grid-s1-dir', L_VAL_G,
         '--data-dir', L_DATA, '--split', constants.V2_SPLIT_NEXAR_VAL, '--out-dir', local],
        log=LOCAL / f'nexar_val_{row}.log')
    publish(local, out)
    r = json.loads((out / 'nexar_readout.json').read_text())
    print(f"nexar_val {row}: crop endpoint per seed {r['per_run_crop_endpoint']} ({(time.time() - t0) / 60:.1f} min)")
flush_drive()

## 5. **DoTA-CAP-dev** — protocol B per row (S2-7, GPU)

`s1[::3]`, whole clip, per-clip min-max, native frames, macro over two-class clips; input = v2's A2 bake of
CAP-dev with the **Nexar** statistics + Stage 0's V7 grids (R5Δ's change is taken inside each clip, ℓ = 6).
Writes per row `clip_aucs.json` (seed-averaged) and `per_run_macro` (paired by seed).

In [ ]:
CAP_IDS = sorted(load_split(CAP_DEV))
L_DOTA_S1  = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
L_DOTA_A2  = stage(DOTA_A2, LOCAL / 'stage' / 'dota_cap_dev_A2', CAP_IDS)
L_DOTA_G   = stage(DOTA_GRID, LOCAL / 'stage' / 'dota_cap_dev_grid', CAP_IDS)
L_DOTA_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
PB_OUT = ROOT / 'eval' / 'dota' / CAP_DEV
for row in ALL_ROWS:
    out = PB_OUT / row
    if (out / 'protocol_b_readout.json').exists():
        print(f'dota {row}: already read')
        continue
    local, t0 = LOCAL / 'dota' / row, time.time()
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *run_args(row),
         '--input-dir', L_DOTA_A2, '--grid-dir', L_DOTA_G, '--s1-dir', L_DOTA_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LAB, '--split', CAP_DEV, '--out-dir', local, '--device', 'auto'],
        log=LOCAL / f'dota_{row}.log')
    publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    print(f"dota {row}: macro {r['macro']['mean']:.4f} per seed {r['per_run_macro']} ({(time.time() - t0) / 60:.1f} min)")
flush_drive()

## 6. Maps — dump R3 / R5 / R5Δ per seed on CAP-dev (GPU), then the **S2-8** read (CPU)

R5 / R5Δ map `E_t(j) = Σ_m s_{t,m} a_{t,m,j}`, R3 its single query's attention. The read averages hit rates over
seeds per clip, then D3 hit − chance, Amendment 1's lift − lift_pre and hit − centre prior (95 % cluster bootstrap);
"points" iff all three lows > 0. STAUC (§2.4 B3) is printed.

In [ ]:
RAW_ANN = DOTA / 'raw_annotations'
ANN = RAW_ANN / 'annotations'
DOTA_LOCAL = LOCAL / 'dota_meta'
DOTA_LOCAL.mkdir(parents=True, exist_ok=True)
if not ANN.is_dir():
    ANN = DOTA_LOCAL / 'annotations'
    if not ANN.exists():
        subprocess.run(f'cd {DOTA_LOCAL} && curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip',
                       shell=True, check=True)
META = DOTA / 'metadata_val.json'
SPLIT = DOTA_LOCAL / 'val_ids.txt'
SPLIT.write_text(''.join(f'{v}\n' for v in json.loads(META.read_text())))
print('annotations:', len(list(ANN.glob('*.json'))))

MAPS = ROOT / 'maps'
for row in MAP_ROWS:
    for seed in SEEDS:
        out = MAPS / f'maps_{row}_s{seed}.npz'
        if out.exists() and out.with_suffix('.json').exists():
            print(f'maps {row} s{seed}: already dumped')
            continue
        local = LOCAL / 'maps' / out.name
        run([sys.executable, '-m', 'core.tools.v5_maps', 'dump', '--name', f'{row}_s{seed}',
             '--ckpt', run_dir(row, seed) / 'checkpoint_last.pt', '--input-dir', L_DOTA_A2,
             '--grid-dir', L_DOTA_G, '--s1-dir', L_DOTA_S1, '--data-dir', L_DOTA_LAB, '--out', local],
            log=LOCAL / f'maps_{row}_s{seed}.log')
        publish(local.with_suffix('.json'), out.with_suffix('.json'))
        publish(local, out)
flush_drive()

runs = [x for row in MAP_ROWS for s in SEEDS for x in ('--run', row, s, MAPS / f'maps_{row}_s{s}.npz')]
run([sys.executable, '-m', 'core.tools.v5_maps', 'read', *runs, '--metadata', META, '--split-file', SPLIT,
     '--annotations-dir', ANN, '--out-dir', REPORTS])
print((REPORTS / 'maps_readout.md').read_text())

## 7. Read-out (S2-7 / S2-9 / S2-10 + S2-14 prints) → **paste `stage2_readout.md` and `maps_readout.md` to Claude**

In [ ]:
run([sys.executable, '-m', 'core.tools.v5_stage2', '--pb-dir', PB_OUT, '--nexar-dir', NEXAR_OUT,
     '--maps-readout', REPORTS / 'maps_readout.json',
     '--provenance', S1_REPORTS / 'g4_provenance.json', REPORTS / 'g4_provenance.json',
     '--collapse', REPORTS / 'g6_collapse.json', '--runs-root', ROOT, '--prior', PRIOR,
     '--r0-pb', R0_PB, '--a2-pb', A2_PB, '--rows', *NEW_ROWS, '--seeds', *SEEDS, '--out-dir', REPORTS])
DECISION = json.loads((REPORTS / 'stage2_readout.json').read_text())['decision']
print((REPORTS / 'stage2_readout.md').read_text())
flush_drive()
print('Drive flushed -- paste REPORTS/stage2_readout.md + maps_readout.md to Claude')

## 8. **S2-11** `nexar_test` — once, R1 + the adopted row (printed only)

Run only **after** step 7 is written and pasted. Off by default: set `OPEN_NEXAR_TEST = True`. Skipped under NULL /
NOT-ADOPTED (no adopted row). Extracts `nexar_test` V7 grids at `s1` (S2-2: only now), writes the open marker,
then one `nexar_eval --final` per row. Once means once: an existing read is never re-scored.

In [ ]:
OPEN_NEXAR_TEST = False
ADOPTED = DECISION['adopted']
if not OPEN_NEXAR_TEST:
    print('S2-11 not opened (OPEN_NEXAR_TEST = False)')
elif ADOPTED is None:
    print(f"S2-11 skipped: outcome {DECISION['outcome']} has no adopted row")
else:
    TEST_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TEST, final=True))
    run([sys.executable, '-m', 'core.tools.v5_stage1', 'extract', '--root', RAW, '--census', CENSUS,
         '--ids-file', write_ids(LOCAL / 'ids_test.txt', TEST_IDS), '--video-s1-dir', VIDEO_S1,
         '--grid-dir', TEST_GRID, '--stride', constants.V5_NEXAR_VAL_STRIDE,
         '--report', REPORTS / 'extract_test.json', '--batch-size', '32', '--device', 'auto',
         '--out-dir', REPORTS / 'test_extract'], log=LOCAL / 'extract_test.log')
    gate_dir = REPORTS / 'test_extract'
    g = json.loads((gate_dir / f'g1_extract_s{constants.V5_NEXAR_VAL_STRIDE}.json').read_text())
    print((gate_dir / f'g1_extract_s{constants.V5_NEXAR_VAL_STRIDE}.md').read_text())
    assert g['pass'], 'nexar_test grid extraction failed S1-G1 (length) -- paste the report to Claude'
    marker = ROOT / 'eval' / 'nexar_test' / constants.NEXAR_TEST_MARKER
    if not marker.exists():
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps({'opened_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                      'rule': 'PREREG_V5 S2-11', 'rows': [CONTROL, ADOPTED],
                                      'seeds': SEEDS, 'outcome': DECISION['outcome']}, indent=2))
        print('nexar_test OPENED now:', marker)
    L_TEST_X = stage(CLIP_S1, LOCAL / 'stage' / 'clip_s1_test', TEST_IDS)
    L_TEST_U = stage(VIDEO_S1, LOCAL / 'stage' / 'video_s1_test', TEST_IDS)
    L_TEST_G = stage(TEST_GRID, LOCAL / 'stage' / 'grid_s1_test', TEST_IDS)
    for row in (CONTROL, ADOPTED):
        out = ROOT / 'eval' / 'nexar_test' / row
        if (out / 'nexar_readout.json').exists():
            print(f'nexar_test {row}: already read -- not re-scored')
            continue
        local = LOCAL / 'nexar_test' / row
        run([sys.executable, '-m', 'core.tools.nexar_eval', *run_args(row), '--census', CENSUS,
             '--clip-s1-dir', L_TEST_X, '--video-s1-dir', L_TEST_U, '--stats-dir', L_A2,
             '--grid-s1-dir', L_TEST_G, '--data-dir', L_DATA, '--split', constants.V2_SPLIT_NEXAR_TEST,
             '--final', '--out-dir', local], log=LOCAL / f'nexar_test_{row}.log')
        publish(local, out)
        r = json.loads((out / 'nexar_readout.json').read_text())
        print(f"nexar_test {row}: crop endpoint per seed {r['per_run_crop_endpoint']}")
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- safe to close the runtime')